# Trustworthy RAG Preference Alignment with DPO
### Model: Qwen3-4B-Instruct-2507
### Dataset: declare-lab/Trust-Data

Notebook 實作流程架構：

1. 固定模型與資料集 revision，便於日後重現。
2. 載入 Trust-Data 官方 `trust_align_dpo` preference pairs。
3. 將資料轉成 TRL 的 conversational preference 格式。
4. 做 token 長度檢查，排除超長 pair，避免截斷回答或引用。
5. 載入前一階段 Trust-Align SFT 的 LoRA adapter。
6. 以 SFT 起點建立 DPO policy 與固定 reference，再執行 4-bit QLoRA DPO。
7. 比較 DPO 前後的輸出、eval loss、引用格式與拒答行為。
8. 儲存包含 SFT + DPO 效果的可重載 LoRA adapter。

> 第一次執行使用 `PROFILE = "smoke_test"`。整條流程成功後，可再改成 `colab_project`或`full_dataset`。

### 實作定位

預設 `START_FROM_SFT = True`，流程為：

```text
Qwen3 base + Trust-Align SFT adapter → DPO policy/reference → SFT+DPO adapter
```

## 1. 安裝固定版本套件

固定版本是為了讓 Notebook 日後仍有較高機率可以重現。若 Colab 顯示已載入舊版
`transformers`，安裝後請重新啟動 runtime，再從下一個 cell 開始執行。

In [ ]:
%pip install -q --upgrade \
  "transformers==5.17.0" \
  "trl==1.12.0" \
  "peft==0.20.0" \
  "datasets==5.0.1" \
  "accelerate==1.14.0" \
  "bitsandbytes==0.50.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 178.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.6 MB/s eta 0:00:00


## 2. 環境檢查與共用設定

In [ ]:
import gc
import json
import os
import random
import re
import shutil
from pathlib import Path

import bitsandbytes as bnb
import datasets
import numpy as np
import pandas as pd
import peft
import torch
import transformers
import trl
from datasets import load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import DPOConfig, DPOTrainer

assert torch.cuda.is_available(), (
    "找不到 CUDA GPU。請在 Colab 選擇 Runtime → Change runtime type → GPU。"
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

gpu_name = torch.cuda.get_device_name(0)
gpu_major, gpu_minor = torch.cuda.get_device_capability(0)
use_bf16 = gpu_major >= 8  # A100、L4 等；T4 使用 fp16
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"GPU: {gpu_name}")
print(f"Compute capability: {gpu_major}.{gpu_minor}")
print(f"Training dtype: {compute_dtype}")
print(
    {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "peft": peft.__version__,
        "datasets": datasets.__version__,
        "bitsandbytes": bnb.__version__,
    }
)

from google.colab import drive
drive.mount("/content/drive")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Compute capability: 12.0
Training dtype: torch.bfloat16
{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'trl': '1.12.0', 'peft': '0.20.0', 'datasets': '5.0.1', 'bitsandbytes': '0.50.2'}
Mounted at /content/drive


## 3. 實驗設定

可使用三種 profile：

| Profile | 訓練資料 | 用途 |
|---|---:|---|
| `smoke_test` | 最多 256 pairs、30 steps | 第一次驗證程式與顯存 |
| `colab_project` | 最多 4,000 pairs、1 epoch | Colab Pro 主要作品集實驗 |
| `full_dataset` | 全部可用 pairs、1 epoch | A100 或較長執行時間 |

這裡預設 2,048 tokens，是因為原始 Trust-Align DPO 設定同樣使用 2,048，且 DPO
每筆資料要同時計算 chosen 與 rejected。A100 上可以另外實驗 3,072 或 4,096，
但請重新觀察顯存與訓練時間。

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"

DATASET_ID = "declare-lab/Trust-Data"
DATASET_CONFIG = "trust_align_dpo"
DATASET_REVISION = "0b89ac014bdcde23c083592af8fe6ce563b128b7"

# 正式 SFT → DPO pipeline 請保持 True。
START_FROM_SFT = True
SFT_ADAPTER_PATH = "/content/qwen3-4b-trust-data-sft-lora"

# 若 SFT zip 已放在 Google Drive，可把此值改成 True 自動掛載並解壓縮。
LOAD_SFT_ZIP_FROM_DRIVE = True
SFT_ADAPTER_ZIP = (
    "/content/drive/MyDrive/llm-alignment-artifacts/"
    "qwen3-4b-trust-data-sft-lora.zip"
)

PROFILE = "full_dataset"  # smoke_test | colab_project | full_dataset
MAX_LENGTH = 2048
RUN_DIR = "/content/qwen3-4b-trust-data-dpo-run"
ADAPTER_DIR = (
    "/content/qwen3-4b-trust-data-sft-dpo-lora"
    if START_FROM_SFT
    else "/content/qwen3-4b-trust-data-dpo-lora"
)

PROFILES = {
    "smoke_test": {
        "train_limit": 256,
        "eval_limit": 64,
        "max_steps": 30,
        "num_train_epochs": 1.0,
        "eval_steps": 15,
        "generation_eval_n": 4,
    },
    "colab_project": {
        "train_limit": 4000,
        "eval_limit": 256,
        "max_steps": -1,
        "num_train_epochs": 1.0,
        "eval_steps": 100,
        "generation_eval_n": 16,
    },
    "full_dataset": {
        "train_limit": None,
        "eval_limit": 512,
        "max_steps": -1,
        "num_train_epochs": 3.0,
        "eval_steps": 250,
        "generation_eval_n": 24,
    },
}

if PROFILE not in PROFILES:
    raise ValueError(f"未知 PROFILE：{PROFILE}")

profile_cfg = PROFILES[PROFILE]
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

print(
    json.dumps(
        {
            "profile": PROFILE,
            "start_from_sft": START_FROM_SFT,
            "sft_adapter_path": SFT_ADAPTER_PATH if START_FROM_SFT else None,
            **profile_cfg,
        },
        indent=2,
        ensure_ascii=False,
    )
)

{
  "profile": "full_dataset",
  "start_from_sft": true,
  "sft_adapter_path": "/content/qwen3-4b-trust-data-sft-lora",
  "train_limit": null,
  "eval_limit": 512,
  "max_steps": -1,
  "num_train_epochs": 3.0,
  "eval_steps": 250,
  "generation_eval_n": 24
}


## 4. 準備並驗證 SFT adapter

若前一階段已把 zip 存入 Google Drive，可設 `LOAD_SFT_ZIP_FROM_DRIVE=True`。
解壓後，`SFT_ADAPTER_PATH` 目錄中必須直接看得到 `adapter_config.json`，不能再多包
一層同名資料夾。

In [ ]:
if START_FROM_SFT and LOAD_SFT_ZIP_FROM_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    if not Path(SFT_ADAPTER_ZIP).is_file():
        raise FileNotFoundError(f"找不到 SFT zip：{SFT_ADAPTER_ZIP}")
    Path(SFT_ADAPTER_PATH).mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(SFT_ADAPTER_ZIP, SFT_ADAPTER_PATH)
    print(f"SFT adapter 已解壓縮至：{SFT_ADAPTER_PATH}")

sft_manifest = None
if START_FROM_SFT:
    adapter_config_path = Path(SFT_ADAPTER_PATH) / "adapter_config.json"
    if not adapter_config_path.is_file():
        raise FileNotFoundError(
            "找不到 SFT adapter_config.json。請先執行 SFT Notebook，並將 adapter "
            f"解壓縮到 {SFT_ADAPTER_PATH}。"
        )

    with open(adapter_config_path) as file:
        sft_adapter_config = json.load(file)

    configured_base = sft_adapter_config.get("base_model_name_or_path")
    if configured_base and configured_base != MODEL_ID:
        raise ValueError(
            f"SFT adapter 的 base model 是 {configured_base}，但本 Notebook 設定為 "
            f"{MODEL_ID}。"
        )

    manifest_path = Path(SFT_ADAPTER_PATH) / "trust_align_training_manifest.json"
    if manifest_path.is_file():
        with open(manifest_path) as file:
            sft_manifest = json.load(file)
        if sft_manifest.get("base_model_revision") != MODEL_REVISION:
            raise ValueError("SFT 與 DPO 使用的 base model revision 不一致。")
        if sft_manifest.get("pipeline_stage") != "trust_align_sft":
            raise ValueError("SFT manifest 的 pipeline_stage 不正確。")

    print("SFT adapter 驗證完成。")
    print(json.dumps(sft_manifest or sft_adapter_config, indent=2, ensure_ascii=False))
else:
    print("START_FROM_SFT=False：將執行原始 Qwen3 的 direct DPO 對照實驗。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SFT adapter 已解壓縮至：/content/qwen3-4b-trust-data-sft-lora
SFT adapter 驗證完成。
{
  "pipeline_stage": "trust_align_sft",
  "base_model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "base_model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "dataset_id": "declare-lab/Trust-Data",
  "dataset_config": "trust_align_sft",
  "dataset_revision": "0b89ac014bdcde23c083592af8fe6ce563b128b7",
  "profile": "full_dataset",
  "max_length": 2048,
  "assistant_only_loss": true,
  "lora": {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
      "down_proj",
      "gate_proj",
      "k_proj",
      "o_proj",
      "q_proj",
      "up_proj",
      "v_proj"
    ]
  }
}


## 5. 載入並檢查 Trust-Data

`trust_align_dpo` 的官方欄位是：

- `prompt`：指令、問題與檢索文件
- `chosen`：較可信任的回答
- `rejected`：較不可信任的回答
- `messages`：prompt 與 chosen 的 ChatML 表示

本實作使用官方 train/test split，不從 train 中隨機切出測試集。

In [ ]:
raw_dataset = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    revision=DATASET_REVISION,
)

print(raw_dataset)
print("train columns:", raw_dataset["train"].column_names)
print("test columns:", raw_dataset["test"].column_names)

required_columns = {"prompt", "chosen", "rejected"}
for split_name in ["train", "test"]:
    missing = required_columns - set(raw_dataset[split_name].column_names)
    if missing:
        raise ValueError(f"{split_name} 缺少必要欄位：{sorted(missing)}")

sample = raw_dataset["train"][0]
print("\nPrompt preview:\n", sample["prompt"][:1200])
print("\nChosen:\n", sample["chosen"])
print("\nRejected:\n", sample["rejected"])

README.md:   0%|          | 0.00/5.04k [00:00<?, ?B/s]

Trust-Align/dpo/train.json: reconstructing file:   0%|          |  0.00B /  137MB            

Trust-Align/dpo/train.json: downloading bytes:           |  0.00B            

Trust-Align/dpo/test.json: reconstructing file:   0%|          |  0.00B / 15.3MB            

Trust-Align/dpo/test.json: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'rejected', 'chosen', 'messages'],
        num_rows: 15598
    })
    test: Dataset({
        features: ['prompt', 'rejected', 'chosen', 'messages'],
        num_rows: 1734
    })
})
train columns: ['prompt', 'rejected', 'chosen', 'messages']
test columns: ['prompt', 'rejected', 'chosen', 'messages']

Prompt preview:
 Instruction: Write an accurate, engaging, and concise answer for the given question using only the provided search results (some of which might be irrelevant) and cite them properly. Use an unbiased and journalistic tone. Always cite for any factual claim. When citing several search results, use [1][2][3]. Cite at least one document and at most three documents in each sentence. If multiple documents support the sentence, only cite a minimum sufficient subset of the documents. If none of the provided documents contain the answer, only respond with "I apologize, but I couldn't find an answer to your question in

## 6. 載入 Qwen tokenizer，建立 conversational preference pairs

不直接把三個欄位當成一般字串，是因為 Qwen Instruct 應使用自己的 chat template。
TRL 接收到 messages 格式後，會正確加入 Qwen 的 user/assistant control tokens。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def select_candidates(dataset_split, final_limit, seed):
    """先多抽一些候選資料，因為後面還要排除超長 pair。"""
    shuffled = dataset_split.shuffle(seed=seed)
    if final_limit is None:
        return shuffled
    candidate_count = min(len(shuffled), max(final_limit * 2, final_limit + 128))
    return shuffled.select(range(candidate_count))


def to_conversational(example):
    return {
        "prompt": [{"role": "user", "content": example["prompt"]}],
        "chosen": [{"role": "assistant", "content": example["chosen"]}],
        "rejected": [{"role": "assistant", "content": example["rejected"]}],
    }


def add_pair_token_lengths(example):
    chosen_out = tokenizer.apply_chat_template(
        example["prompt"] + example["chosen"],
        tokenize=True,
        add_generation_prompt=False,
    )
    rejected_out = tokenizer.apply_chat_template(
        example["prompt"] + example["rejected"],
        tokenize=True,
        add_generation_prompt=False,
    )

    # 若回傳的是 dict / BatchEncoding，取 input_ids 的長度；若回傳 list 則直接取 len
    chosen_ids = chosen_out["input_ids"] if "input_ids" in chosen_out else chosen_out
    rejected_ids = rejected_out["input_ids"] if "input_ids" in rejected_out else rejected_out

    return {
        "chosen_tokens": len(chosen_ids),
        "rejected_tokens": len(rejected_ids),
    }


def prepare_split(dataset_split, final_limit, seed):
    candidates = select_candidates(dataset_split, final_limit, seed)
    prepared = candidates.map(
        to_conversational,
        remove_columns=candidates.column_names,
        desc="Convert to conversational DPO format",
    )
    prepared = prepared.map(
        add_pair_token_lengths,
        desc="Count pair tokens",
    )

    before_filter = len(prepared)
    prepared = prepared.filter(
        lambda row: max(row["chosen_tokens"], row["rejected_tokens"])
        <= MAX_LENGTH,
        desc=f"Keep complete pairs <= {MAX_LENGTH} tokens",
    )
    removed = before_filter - len(prepared)

    if final_limit is not None:
        prepared = prepared.select(range(min(final_limit, len(prepared))))

    print(
        f"候選 {before_filter:,} 筆；排除超長 {removed:,} 筆；"
        f"最後保留 {len(prepared):,} 筆"
    )
    return prepared


train_prepared = prepare_split(
    raw_dataset["train"], profile_cfg["train_limit"], SEED
)
eval_prepared = prepare_split(
    raw_dataset["test"], profile_cfg["eval_limit"], SEED + 1
)

if len(train_prepared) == 0 or len(eval_prepared) == 0:
    raise RuntimeError("長度過濾後沒有資料，請提高 MAX_LENGTH。")

length_summary = pd.DataFrame(
    {
        "train_chosen": pd.Series(train_prepared["chosen_tokens"]),
        "train_rejected": pd.Series(train_prepared["rejected_tokens"]),
    }
).describe(percentiles=[0.5, 0.9, 0.95, 0.99])
display(length_summary)

# 保留一份含 raw messages 與 token length 的版本供後續比較；
# 傳給 Trainer 的資料只留下必要欄位。
eval_view = eval_prepared
columns_to_remove = ["chosen_tokens", "rejected_tokens"]
train_dataset = train_prepared.remove_columns(columns_to_remove)
eval_dataset = eval_prepared.remove_columns(columns_to_remove)

display(train_dataset[0])

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Count pair tokens:   0%|          | 0/1024 [00:00<?, ? examples/s]

Keep complete pairs <= 2048 tokens:   0%|          | 0/1024 [00:00<?, ? examples/s]

候選 1,024 筆；排除超長 0 筆；最後保留 512 筆


,train_chosen,train_rejected
count,15598.000000,15598.000000
mean,972.876523,981.258943
std,68.135289,75.073598
min,771.000000,777.000000
50%,966.000000,970.000000
90%,1052.000000,1066.000000
95%,1088.000000,1117.000000
99%,1190.030000,1248.000000
max,1561.000000,1658.000000


{'prompt': [{'content': 'Instruction: Write an accurate, engaging, and concise answer for the given question using only the provided search results (some of which might be irrelevant) and cite them properly. Use an unbiased and journalistic tone. Always cite for any factual claim. When citing several search results, use [1][2][3]. Cite at least one document and at most three documents in each sentence. If multiple documents support the sentence, only cite a minimum sufficient subset of the documents. If none of the provided documents contain the answer, only respond with "I apologize, but I couldn\'t find an answer to your question in the search results.".\n\nQuestion: When does the revolutionary war start and end?\n\nDocument [1](Title: Massachusetts National Guard): to an emergency - theoretically on a minute\'s notice. In the early morning hours of April 19, 1775, the militia company of Lexington, commanded by Captain John Parker, confronted British forces heading to Concord to sear

## 7. 建立 4-bit QLoRA 與 DPO 設定

- 利用 NF4 + double quantization 降低 base model 顯存。
- `START_FROM_SFT=True` 時會繼續更新既有 SFT adapter，不會建立全新的 LoRA。
- `ref_model=None` 時，TRL 會自動把 DPO 開始前的 SFT policy 當作 reference。
  對既有 PEFT model，TRL 會建立凍結的 reference adapter，無須常駐第二份 4B base。
- `beta=0.5` 參考原始 Trust-Align DPO 設定。
- 原論文採 full fine-tuning、learning rate `5e-7`；這裡只訓練 LoRA，所以使用
  `5e-6`。

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = DPOConfig(
    output_dir=RUN_DIR,
    max_length=MAX_LENGTH,
    truncation_mode="keep_start",
    beta=0.5,
    loss_type=["sigmoid"],
    learning_rate=5e-6,
    num_train_epochs=profile_cfg["num_train_epochs"],
    max_steps=profile_cfg["max_steps"],
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    # Transformers 5 將小於 1 的 warmup_steps 解讀為總步數比例。
    warmup_steps=0.05,
    max_grad_norm=1.0,
    bf16=use_bf16,
    fp16=not use_bf16,
    tf32=use_bf16,
    logging_strategy="steps",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=profile_cfg["eval_steps"],
    save_strategy="steps",
    save_steps=profile_cfg["eval_steps"],
    save_total_limit=2,
    save_only_model=True,
    report_to="none",
    dataset_num_proc=1,
    dataloader_num_workers=0,
    remove_unused_columns=True,
    seed=SEED,
    data_seed=SEED,
)

print(training_args)

DPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.5,
bf16=True,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=1,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=Fals

## 8. 載入 SFT policy 並建立 DPOTrainer

這一步會先載入 4-bit Qwen3 base，再掛載 SFT adapter 並設為可訓練。

In [ ]:
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=compute_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)

if START_FROM_SFT:
    # is_trainable=True 讓 default adapter 成為 DPO policy。
    model_for_dpo = PeftModel.from_pretrained(
        base_model,
        SFT_ADAPTER_PATH,
        adapter_name="default",
        is_trainable=True,
    )
    trainer_peft_config = None
else:
    # 保留 direct-DPO 對照實驗：從原始 base 建立新的 LoRA adapter。
    model_for_dpo = base_model
    trainer_peft_config = lora_config

trainer = DPOTrainer(
    model=model_for_dpo,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=trainer_peft_config,
)

if START_FROM_SFT and "ref" not in trainer.model.peft_config:
    raise RuntimeError("TRL 沒有建立凍結的 SFT reference adapter。")

# TRL 的目前版本會把 QLoRA adapter 轉成 bf16；T4 沒有原生 bf16，
# 因此在 T4 類型 GPU 上恢復成標準的 fp32 LoRA weights，forward 時再由
# mixed-precision autocast 處理。L4/A100 則保留 bf16 adapter。
if not use_bf16:
    for parameter in trainer.model.parameters():
        if parameter.requires_grad:
            parameter.data = parameter.data.to(torch.float32)

if START_FROM_SFT:
    # TRL 建立 ref 後會依 QLoRA 策略調整可訓練 adapter dtype。這裡在最終
    # dtype 確定後再同步一次，確保 DPO 的 policy 與 reference 從同一 SFT 狀態出發。
    named_parameters = dict(trainer.model.named_parameters())
    policy_parameters = {
        name: parameter
        for name, parameter in named_parameters.items()
        if ".default." in name
    }
    if not policy_parameters:
        raise RuntimeError("找不到可訓練的 default SFT adapter parameters。")

    for policy_name, policy_parameter in policy_parameters.items():
        ref_name = policy_name.replace(".default.", ".ref.")
        if ref_name not in named_parameters:
            raise RuntimeError(f"reference adapter 缺少參數：{ref_name}")
        ref_parameter = named_parameters[ref_name]
        ref_parameter.data = policy_parameter.detach().clone()
        ref_parameter.requires_grad_(False)

    trainer.model.set_adapter("default")
    ref_parameters = [
        parameter
        for name, parameter in trainer.model.named_parameters()
        if ".ref." in name
    ]
    if not ref_parameters or any(parameter.requires_grad for parameter in ref_parameters):
        raise RuntimeError("DPO reference adapter 不存在或沒有被凍結。")
    print("DPO policy：可訓練的 default（SFT）adapter")
    print("DPO reference：凍結的 ref（SFT snapshot）adapter")

trainer.model.print_trainable_parameters()
print(
    f"Allocated GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)
print(
    f"Reserved GPU memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
)

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/15598 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset (num_proc=1):   0%|          | 0/15598 [00:00<?, ? exampl…

Tokenizing eval dataset (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/…

DPO policy：可訓練的 default（SFT）adapter
DPO reference：凍結的 ref（SFT snapshot）adapter
trainable params: 33,030,144 || all params: 4,088,528,384 || trainable%: 0.8079
Allocated GPU memory: 2.62 GiB
Reserved GPU memory: 2.78 GiB


## 9. DPO 前的 held-out generation

`START_FROM_SFT=True` 時，這裡是 **SFT 後、DPO 前** 的輸出；DPO 完成後會使用
完全相同的 prompts 比較。如果是 direct-DPO 對照組，則是原始 Qwen3 的輸出。

In [ ]:
REFUSAL_TEXT = (
    "I apologize, but I couldn't find an answer to your question "
    "in the search results."
)


def generate_response(model, prompt_text, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt_text}]
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(rendered, return_tensors="pt")
    model_inputs = {key: value.to(model.device) for key, value in model_inputs.items()}

    model.eval()
    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0, model_inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


generation_n = min(profile_cfg["generation_eval_n"], len(eval_view))


def target_requires_refusal(chosen_text):
    normalized = " ".join(chosen_text.lower().split())
    return (
        "couldn't find an answer" in normalized
        or "could not find an answer" in normalized
    )


# 儘量各取一半「應回答」與「應拒答」資料，讓小型 smoke test 仍能觀察兩種行為。
refusal_indices = [
    index
    for index in range(len(eval_view))
    if target_requires_refusal(eval_view[index]["chosen"][0]["content"])
]
answer_indices = [
    index
    for index in range(len(eval_view))
    if not target_requires_refusal(eval_view[index]["chosen"][0]["content"])
]
target_refusal_n = generation_n // 2
selected_indices = (
    refusal_indices[:target_refusal_n]
    + answer_indices[: generation_n - target_refusal_n]
)
if len(selected_indices) < generation_n:
    used = set(selected_indices)
    selected_indices.extend(
        index
        for index in range(len(eval_view))
        if index not in used
    )
selected_indices = selected_indices[:generation_n]

generation_records = []
for index in selected_indices:
    row = eval_view[index]
    generation_records.append(
        {
            "index": index,
            "prompt": row["prompt"][0]["content"],
            "chosen": row["chosen"][0]["content"],
            "rejected": row["rejected"][0]["content"],
        }
    )

baseline_outputs = [
    generate_response(trainer.model, row["prompt"])
    for row in generation_records
]

baseline_preview = pd.DataFrame(
    {
        "chosen": [row["chosen"] for row in generation_records],
        "rejected": [row["rejected"] for row in generation_records],
        "starting_policy_output": baseline_outputs,
    }
)
display(baseline_preview)

,chosen,rejected,starting_policy_output
0,"I apologize, but I couldn't find an answer to ...",The uppermost layer of the skin is known as th...,The uppermost layer of the skin is the epiderm...
1,"I apologize, but I couldn't find an answer to ...","The song ""If I Could Turn Back Time"" was sung ...","The song ""If I Could Turn Back Time"" is sung b..."
2,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","To identify unknown bodies by dental records, ..."
3,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
4,"I apologize, but I couldn't find an answer to ...","When you see faces in everything, it is known ...","I apologize, but I couldn't find an answer to ..."
5,"I apologize, but I couldn't find an answer to ...","The winner of ""I'm a Celebrity...Get Me Out of...","I apologize, but I couldn't find an answer to ..."
6,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","The song ""Diamonds Are a Girl's Best Friend"" i..."
7,"I apologize, but I couldn't find an answer to ...",The next Olympics in 2018 will be held in Pyeo...,The next Olympics in 2018 will be held in Rio ...
8,"I apologize, but I couldn't find an answer to ...","Tongue N' Cheek [3], Jus' a Rascal [5], Maths ...","Tongue N [3], Jus' a Rascal [5], The Fifth [2]..."
9,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...",Drinking saltwater is detrimental to our healt...


## 10. 訓練前 DPO evaluation

未訓練的 policy 與 reference 相同，初始 DPO loss 通常接近
$-\log(0.5) \approx 0.693$。sanity check，不代表模型回答品質為 0.693。

In [ ]:
initial_eval_metrics = trainer.evaluate(metric_key_prefix="initial_eval")
initial_eval_metrics

Training Loss,Validation Loss,Step,Eval Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,No log,0,0.693147,0.582252,0.000000,-1.125867,-0.874069,0.841004,0.000000,0.000000,0.000000,0.000000,-20.441798,-26.462678


{'initial_eval_loss': 0.6931473016738892,
 'eval_entropy': 0.5822521626232628,
 'eval_num_tokens': 0.0,
 'eval_logits/chosen': -1.1258668888921552,
 'eval_logits/rejected': -0.8740687993854523,
 'eval_mean_token_accuracy': 0.8410041946335696,
 'eval_rewards/chosen': 0.0,
 'eval_rewards/rejected': 0.0,
 'eval_rewards/accuracies': 0.0,
 'eval_rewards/margins': 0.0,
 'eval_logps/chosen': -20.441798016428947,
 'eval_logps/rejected': -26.462678303942084}

## 11. 開始 DPO 訓練

此 cell 請只執行一次。若 runtime 中斷，可把 `resume_from_checkpoint` 改成最近的
checkpoint 路徑。

In [ ]:
train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

# 只輸出 policy 的 default adapter，不把 frozen reference 與 checkpoints 一起打包。
# 此 adapter 的權重已包含 SFT 起點以及後續 DPO 更新。
trainer.model.save_pretrained(
    ADAPTER_DIR,
    selected_adapters=["default"],
    safe_serialization=True,
)
tokenizer.save_pretrained(ADAPTER_DIR)

dpo_manifest = {
    "pipeline_stage": "trust_align_dpo",
    "start_from_sft": START_FROM_SFT,
    "base_model_id": MODEL_ID,
    "base_model_revision": MODEL_REVISION,
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "sft_manifest": sft_manifest,
    "profile": PROFILE,
    "max_length": MAX_LENGTH,
    "beta": training_args.beta,
}
with open(f"{ADAPTER_DIR}/trust_align_training_manifest.json", "w") as file:
    json.dump(dpo_manifest, file, indent=2, ensure_ascii=False)

stage_label = "SFT+DPO" if START_FROM_SFT else "direct-DPO"
print(f"訓練 checkpoints：{RUN_DIR}")
print(f"可部署的 {stage_label} LoRA adapter：{ADAPTER_DIR}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
250,0.397834,0.359198,0.651263,3910286.000000,-1.125184,-0.916560,0.843202,0.012817,-1.234333,0.816406,1.247150,-20.416164,-28.931344
500,0.164496,0.267787,0.706208,7818444.000000,-1.264067,-1.059021,0.839090,-0.608123,-3.705462,0.818359,3.097340,-21.658043,-33.873603
750,0.198935,0.223546,0.760959,11723261.000000,-1.259592,-1.080164,0.839017,-0.551256,-4.533342,0.841797,3.982086,-21.544311,-35.529363
1000,0.053780,0.201737,0.827723,15633677.000000,-1.347382,-1.182870,0.836632,-0.848697,-6.144807,0.847656,5.296110,-22.139192,-38.752292
1250,0.144996,0.199586,0.800930,19534811.000000,-1.462249,-1.280088,0.832732,-1.170510,-7.107225,0.849609,5.936716,-22.782817,-40.677129
1500,0.273054,0.211119,0.870636,23445241.000000,-1.462537,-1.283458,0.825578,-1.736194,-8.574560,0.851562,6.838366,-23.914185,-43.611798
1750,0.179854,0.199291,0.926523,27357818.000000,-1.479823,-1.301820,0.819036,-2.409601,-9.444373,0.857422,7.034772,-25.261001,-45.351425
2000,0.121405,0.184849,0.975269,31259728.000000,-1.524051,-1.353638,0.817103,-2.167937,-9.356706,0.857422,7.188768,-24.777673,-45.176090
2250,0.185072,0.189634,0.930108,35168070.000000,-1.515816,-1.338979,0.811428,-2.417262,-9.719592,0.853516,7.302330,-25.276323,-45.901863
2500,0.397622,0.186401,0.949489,39070334.000000,-1.544720,-1.378975,0.813490,-2.610407,-10.913342,0.857422,8.302935,-25.662612,-48.289363


***** train metrics *****
  epoch                    =          3.0
  total_flos               = 1909359384GF
  train_loss               =       0.2006
  train_runtime            =   7:52:59.63
  train_samples_per_second =        1.649
  train_steps_per_second   =        0.206
訓練 checkpoints：/content/qwen3-4b-trust-data-dpo-run
可部署的 SFT+DPO LoRA adapter：/content/qwen3-4b-trust-data-sft-dpo-lora


## 12. 訓練後評估與前後比較

In [ ]:
final_eval_metrics = trainer.evaluate(metric_key_prefix="final_eval")


def remove_metric_prefix(metrics, prefix):
    expected = f"{prefix}_"
    return {
        (key[len(expected) :] if key.startswith(expected) else key): value
        for key, value in metrics.items()
    }


metric_comparison = pd.DataFrame(
    [
        remove_metric_prefix(initial_eval_metrics, "initial_eval"),
        remove_metric_prefix(final_eval_metrics, "final_eval"),
    ],
    index=["before_dpo", "after_dpo"],
).T
display(metric_comparison)

aligned_outputs = [
    generate_response(trainer.model, row["prompt"])
    for row in generation_records
]

comparison_df = pd.DataFrame(
    {
        "index": [row["index"] for row in generation_records],
        "chosen": [row["chosen"] for row in generation_records],
        "rejected": [row["rejected"] for row in generation_records],
        "before_dpo": baseline_outputs,
        "after_dpo": aligned_outputs,
    }
)

comparison_path = f"{ADAPTER_DIR}/heldout_generation_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)
print(f"比較結果：{comparison_path}")

Training Loss,Validation Loss,Step,Eval Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.114695,No log,5850,0.182966,0.943206,91441815.000000,-1.527590,-1.362979,0.808474,-3.080253,-11.121288,0.855469,8.041035,-26.602305,-48.705255


,before_dpo,after_dpo
loss,0.693147,1.829663e-01
eval_entropy,0.582252,9.432062e-01
eval_num_tokens,0.000000,9.144182e+07
eval_logits/chosen,-1.125867,-1.527590e+00
eval_logits/rejected,-0.874069,-1.362979e+00
eval_mean_token_accuracy,0.841004,8.084740e-01
eval_rewards/chosen,0.000000,-3.080253e+00
eval_rewards/rejected,0.000000,-1.112129e+01
eval_rewards/accuracies,0.000000,8.554688e-01
eval_rewards/margins,0.000000,8.041035e+00


,index,chosen,rejected,before_dpo,after_dpo
0,0,"I apologize, but I couldn't find an answer to ...",The uppermost layer of the skin is known as th...,The uppermost layer of the skin is the epiderm...,"I apologize, but I couldn't find an answer to ..."
1,1,"I apologize, but I couldn't find an answer to ...","The song ""If I Could Turn Back Time"" was sung ...","The song ""If I Could Turn Back Time"" is sung b...","[1][3]. The song ""Turn Back Time"" is performed..."
2,5,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","To identify unknown bodies by dental records, ...","I apologize, but I couldn't find an answer to ..."
3,14,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
4,15,"I apologize, but I couldn't find an answer to ...","When you see faces in everything, it is known ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
5,39,"I apologize, but I couldn't find an answer to ...","The winner of ""I'm a Celebrity...Get Me Out of...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
6,43,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","The song ""Diamonds Are a Girl's Best Friend"" i...","I apologize, but I couldn't find an answer to ..."
7,46,"I apologize, but I couldn't find an answer to ...",The next Olympics in 2018 will be held in Pyeo...,The next Olympics in 2018 will be held in Rio ...,"I apologize, but I couldn't find an answer to ..."
8,52,"I apologize, but I couldn't find an answer to ...","Tongue N' Cheek [3], Jus' a Rascal [5], Maths ...","Tongue N [3], Jus' a Rascal [5], The Fifth [2]...","Maths [4], Tongue N [3]."
9,55,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...",Drinking saltwater is detrimental to our healt...,The body's regulation of salt and water balanc...


比較結果：/content/qwen3-4b-trust-data-sft-dpo-lora/heldout_generation_comparison.csv


## 13. 簡易指標評估：引用與拒答

下列指標適合做 training sanity check，但**不是官方 Trust-Score**：

- `citation_presence_non_refusal`：應回答資料中，輸出是否含 `[n]`。
- `citation_id_validity`：輸出引用編號是否都存在於 prompt 文件。
- `refusal_accuracy`：輸出是否與 chosen 的「應拒答／應回答」類別一致。

它們不判斷 citation 是否真正支持 claim。完整專案仍應使用 Trust-Eval，在 ASQA、
QAMPARI、ELI5、ExpertQA 上評估 response correctness、citation recall/precision 與
refusal groundedness。

In [ ]:
def normalize_text(text):
    return " ".join(text.lower().split())


def is_refusal(text):
    normalized = normalize_text(text)
    return (
        "couldn't find an answer" in normalized
        or "could not find an answer" in normalized
    )


def cited_ids(text):
    return {int(value) for value in re.findall(r"\[(\d+)\]", text)}


def available_document_ids(prompt):
    return {
        int(value)
        for value in re.findall(r"Document\s*\[(\d+)\]", prompt)
    }


def structural_diagnostics(records, outputs):
    rows = []
    for record, output in zip(records, outputs):
        target_refusal = is_refusal(record["chosen"])
        predicted_refusal = is_refusal(output)
        citations = cited_ids(output)
        available = available_document_ids(record["prompt"])
        rows.append(
            {
                "target_refusal": target_refusal,
                "predicted_refusal": predicted_refusal,
                "refusal_correct": target_refusal == predicted_refusal,
                "should_answer": not target_refusal,
                "has_citation": len(citations) > 0,
                "citation_ids_valid": bool(citations) and citations <= available,
            }
        )

    frame = pd.DataFrame(rows)
    answer_rows = frame[frame["should_answer"]]
    return {
        "n": len(frame),
        "refusal_accuracy": frame["refusal_correct"].mean(),
        "citation_presence_non_refusal": (
            answer_rows["has_citation"].mean() if len(answer_rows) else float("nan")
        ),
        "citation_id_validity": (
            answer_rows["citation_ids_valid"].mean()
            if len(answer_rows)
            else float("nan")
        ),
    }


diagnostic_comparison = pd.DataFrame(
    {
        "before_dpo": structural_diagnostics(
            generation_records, baseline_outputs
        ),
        "after_dpo": structural_diagnostics(
            generation_records, aligned_outputs
        ),
    }
)
display(diagnostic_comparison)

,before_dpo,after_dpo
n,24.000000,24.000000
refusal_accuracy,0.541667,0.708333
citation_presence_non_refusal,0.750000,1.000000
citation_id_validity,0.750000,1.000000


## 14. 封裝與下載 LoRA adapter

In [ ]:
archive_base = (
    "/content/qwen3-4b-trust-data-sft-dpo-lora"
    if START_FROM_SFT
    else "/content/qwen3-4b-trust-data-dpo-lora"
)
archive_path = shutil.make_archive(
    archive_base,
    "zip",
    ADAPTER_DIR,
)
print(f"下載檔案：{archive_path}")

# 在 Colab 執行時取消下面兩行註解，即可從瀏覽器下載：
# from google.colab import files
# files.download(archive_path)

下載檔案：/content/qwen3-4b-trust-data-sft-dpo-lora.zip


### 保存到 Google Drive

In [ ]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    # from google.colab import drive

    # drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/llm-alignment-artifacts")
    drive_dir.mkdir(parents=True, exist_ok=True)
    destination = drive_dir / Path(archive_path).name
    shutil.copy2(archive_path, destination)
    print(f"已複製至：{destination}")
else:
    print("SAVE_TO_DRIVE=False；未寫入 Google Drive。")

已複製至：/content/drive/MyDrive/llm-alignment-artifacts/qwen3-4b-trust-data-sft-dpo-lora.zip


### 主要來源

- Dataset: https://huggingface.co/datasets/declare-lab/Trust-Data
- Trust-Align repository: https://github.com/declare-lab/trust-align
- Qwen model card: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507
- TRL DPOTrainer: https://huggingface.co/docs/trl/dpo_trainer